# 5 Long documents: the context window is a budget

In `07_vectors.ipynb` (lesson 1) you met chunking as *mechanics*: a text longer than the
encoder's context window becomes a `(C, D)` matrix instead of a `(D,)` vector, and an
aggregator folds it back. On IMDB that machinery was mostly decorative — 86% of reviews fit
in a single 512-token chunk, so `C` was almost always 1.

This notebook is about chunking as a **decision**. The data is
[SCOTUS from LexGLUE](https://huggingface.co/datasets/coastalcph/lex_glue): US Supreme Court
opinions, labelled with 13 issue areas. The *median* opinion is ~6.500 tokens — thirteen times
the encoder's window — and the longest run past 40.000. When a document does not fit, you have
three options:

1. **Truncate** — read the first 512 tokens and hope the signal is up front. This is what every
   "just use BERT" answer silently does.
2. **Chunk and aggregate** — embed every window, then reduce `(B, C, D) -> (B, D)`.
3. **Pay for a long-context model** — someone else's solution, at someone else's price.

The first two are measurable on a laptop, so we measure them instead of arguing about them.
The question the whole notebook answers: **how much of the document do you actually need to
read?**

In [ ]:
from pathlib import Path
import warnings
import time

import torch
from datasets import load_dataset

warnings.simplefilter("ignore", UserWarning)

ARTEFACTS = Path("../../artefacts").resolve()
ARTEFACTS.mkdir(exist_ok=True)
SCOTUS_DIR = ARTEFACTS / "scotus_probe"
SCOTUS_DIR.mkdir(exist_ok=True)

## 5.1 The data

One row is one opinion: a `text` string and an integer `label` (the issue area, 0-12, from the
[Supreme Court Database](http://scdb.wustl.edu)'s coding). The dataset is parquet-native, so
`load_dataset` just works — no loader scripts, no `data_files` gymnastics.

We work on a **600 train / 300 test subset** so the encoding pass stays in lesson budget
(~4 minutes on Apple Silicon; expect ~15 on a plain CPU). The full 5.000-document cache is a
publish-once job, not a per-student one.

In [ ]:
N_TRAIN, N_TEST = 600, 300

ds = load_dataset("coastalcph/lex_glue", "scotus")
train = ds["train"].shuffle(seed=42).select(range(N_TRAIN))
test = ds["test"].shuffle(seed=42).select(range(N_TEST))

ISSUE_AREAS = [
    "Criminal Procedure", "Civil Rights", "First Amendment", "Due Process",
    "Privacy", "Attorneys", "Unions", "Economic Activity", "Judicial Power",
    "Federalism", "Interstate Relations", "Federal Taxation", "Miscellaneous",
]
K = len(ISSUE_AREAS)

row = train[0]
print(f"{len(train)} train / {len(test)} test opinions, {K} issue areas")
print(f"label {row['label']} = {ISSUE_AREAS[row['label']]}")
print(row["text"][:400])

## 5.2 How badly does this *not* fit?

The encoder is `ibm-granite/granite-embedding-small-english-r2`, the same one that built the
IMDB cache in lesson 1. Before embedding anything, do the one-minute check that decides the
whole notebook: tokenize a few opinions and compare against the window we will use.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "ibm-granite/granite-embedding-small-english-r2"
# MODEL_NAME = "Gerwin/legal-bert-dutch-english"
MAX_LENGTH = 512

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
assert tok is not None
lens = sorted(
    len(tok(t, truncation=False)["input_ids"]) for t in train.select(range(50))["text"]
)
print(f"tokens per opinion (50 docs): median {lens[25]}, min {lens[0]}, max {lens[-1]}")
print(f"window: {MAX_LENGTH} tokens -> a {MAX_LENGTH}-token read sees "
      f"~{MAX_LENGTH / lens[25]:.0%} of the median opinion")

A 512-token window sees less than a tenth of the median opinion. On IMDB the same window
saw the *whole* review 86% of the time. That difference is why this lesson needed a new
dataset.

## 5.3 The cache

Same order of preference as always (conventions §3): load a cache that already exists, build
only if you must. The build branch below is the real thing — `Vectorizer` chunks each opinion
with a sliding window (`stride = max_length // 10`) and embeds every chunk. It is the only
expensive cell in the notebook, and it runs **once**; every later experiment reuses its
output. That asymmetry is the entire point of the cache.

One more thing the loader has to get right, and a hand-rolled `glob` typically does not:
`VectorCache.create`'s folder name is `{timestamp}_{dataset_tag}_{column_name}` -- nothing in
it names the *encoder*. Swap `MODEL_NAME` below and rerun with a naive
`glob(f"*_scotus_{name}_embed")` and you get back the **previous** model's vectors, silently
-- the exact bug [raoulg/vectormesh#1](https://github.com/raoulg/vectormesh/issues/1) is open
to fix in the library. Until it lands, `cache_utils.get_or_create_cache` (this lesson's
version of `2_convolutions/cnn.py` -- one definition, every notebook here imports it) folds
the encoder into the tag itself and raises if a cache's stored `model_tag` ever disagrees with
the vectorizer you handed it.

In [ ]:
from vectormesh import Vectorizer
from vectormesh.data.vectorizers import detect_device

from cache_utils import get_or_create_cache

device = detect_device()


def get_cache(name: str, split):
    vectorizer = Vectorizer(
        model_name=MODEL_NAME, col_name="embed", input_col="text",
        max_length=MAX_LENGTH, device=device,
    )
    return get_or_create_cache(SCOTUS_DIR, f"scotus_{name}", vectorizer, split)


traincache = get_cache("train", train)
testcache = get_cache("test", test)

## 5.4 Read the metadata before writing any model code

The lesson-1 habit, unchanged. Three things to read off it, in order:

- **`hidden_size`** — the `D` of every shape downstream.
- **`tensordtype: 2`** — one item is a rank-2 `(C, D)` matrix, so items have *different
  shapes* and batching needs a padder.
- **`chunk_sizes`** — the histogram that every decision in this notebook comes from.

In [ ]:
assert traincache.metadata is not None
meta = traincache.metadata["embed"]
{k: v for k, v in meta.items() if k != "chunk_sizes"}

In [ ]:
import matplotlib.pyplot as plt

chunks = {int(k): v for k, v in meta["chunk_sizes"].items()}
counts = sorted(chunks)
total = sum(chunks.values())
mean_c = sum(k * v for k, v in chunks.items()) / total

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(counts, [chunks[c] for c in counts], width=1.0)
ax.set_xlabel("chunks per document")
ax.set_ylabel("documents")
ax.set_title(f"{total} opinions: mean {mean_c:.1f} chunks, max {max(counts)}")
plt.show()

for cutoff in (1, 4, 8, 16, 32):
    kept = sum(v for k, v in chunks.items() if k <= cutoff)
    print(f"max_chunks={cutoff:3d}: {total - kept:4d} documents truncated "
          f"({(total - kept) / total:5.1%})")

Compare this against the IMDB histogram from lesson 1: there, one thin bar at `C=1`. Here the
tail runs into the hundreds, so `max_chunks` — the `FixedPadding` parameter you barely noticed
in lesson 1 — has become a real modelling decision: **every value truncates someone**. Pick 16
and you cut into roughly 4 documents in 10; pick 1 and you have reinvented truncation.

Because this cache kept the `text` column, we can look at what a "chunk" actually is before we
start averaging them:

In [ ]:
doc = traincache[3]
emb = torch.tensor(doc["embed"])
stride = meta["stride"]
print(f"opinion with {emb.shape[0]} chunks of {emb.shape[1]} dims "
      f"(window {meta['context_size']}, stride {stride})\n")
enc = tok(doc["text"], truncation=True, max_length=MAX_LENGTH, stride=stride,
          return_overflowing_tokens=True)
for i in (0, len(enc["input_ids"]) // 2):
    snippet = tok.decode(enc["input_ids"][i][:60], skip_special_tokens=True)
    print(f"chunk {i:2d}: {snippet[:180]}...\n")

Chunk 0 is the syllabus — reporter numbers, party names, the case in one paragraph. The middle
chunk is deep inside an argument. Whether the *label* lives up front or is spread through the
document is exactly what the experiment below measures.

## 5.5 From cache to batches

Same two pieces as lesson 1 — `OneHot` for the target, `Collate` for the batch — plus the one
that now matters: `FixedPadding(max_chunks=C)` pads short documents with zero-chunks and
**silently truncates long ones**. `(B, C, D)` on the ladder.

In [ ]:
from torch.utils.data import DataLoader
from vectormesh.components import FixedPadding
from vectormesh.data import Collate, OneHot

onehot = OneHot(num_classes=K, label_col="label", target_col="onehot")
train_oh = traincache.map(onehot)
test_oh = testcache.map(onehot)


def loaders(max_chunks: int):
    collate = Collate(embedding_col="embed", target_col="onehot",
                      padder=FixedPadding(max_chunks=max_chunks))
    return (
        DataLoader(train_oh, batch_size=32, shuffle=True, collate_fn=collate, num_workers=0),
        DataLoader(test_oh, batch_size=32, shuffle=False, collate_fn=collate, num_workers=0),
    )

trainloader, testloader = loaders(16)
X, y = next(iter(trainloader))
X.shape, y.shape

## 5.6 Four axes, not four rungs

The obvious next move is to hand-pick a few pipelines and compare them — truncate, mean,
attention, transformer — the way `02_pretrained` compared a CNN to a frozen head. But this
lesson has **four independent design choices**, not one:

| axis | options |
|---|---|
| aggregator | mean, masked mean, attention, RNN |
| `max_chunks` (how much of the document) | 1, 2, 4, 8, 16, 32 |
| a `TransformerBlock` before aggregating | yes / no |
| the bottleneck width `D` after the embedding | 32, 64, 128, 256, 384 |

Four choices, and $4 \times 6 \times 2 \times 5 = 240$ combinations. Hand-picking four of them
the way a "ladder" does is not a survey of that space, it is four anecdotes. **This is a
hyperparameter search problem**, and the course already has the tool for it —
`ray.tune`, from `4_tuning_networks/01_hypertuner.ipynb`. Nothing new to learn about *ray*
here; the new material is the architecture being searched, not the search machinery.

## 5.7 Tune on a validation split, not on the test set

One discipline point before touching `ray`: whatever configuration the search picks will look
better on the data it was scored against than it really is, simply because the search *chose*
it to look good there. If that data is the test set, the final number you report is not
honest — it is the same leakage goal 3.3 (lesson 3) warns about, one level up: instead of
leaking a row, you leak a *decision*.

So the 600-document train cache is split again — 500 for training each trial, 100 held out as
a **validation** set the search is scored against. The 300-document test cache is not touched
until §5.11, once, after the search is over.

In [ ]:
tune_split = train_oh.train_test_split(test_size=100, seed=42)
tune_train_ds, tune_valid_ds = tune_split["train"], tune_split["test"]
print(f"tune-train {len(tune_train_ds)} / tune-valid {len(tune_valid_ds)} / "
      f"held-out test {len(test_oh)} (untouched until §5.11)")

## 5.8 The search space and the trainable

One environment quirk before `ray` enters the notebook. If your Jupyter session was itself
started with `uv run`, Ray detects that ancestor process and tries to replicate the *exact*
`uv` environment on every worker — which means locating `pyproject.toml` relative to a
`working_dir` it derives from the current directory. That derivation assumes the driver's
working directory *contains* the project, which is false here: this notebook's cwd is
`notebooks/5_attention`, two levels below the repository root, exactly like every other
`../../artefacts` path in this course. The result is a `RuntimeError` about `pyproject.toml`
not being in the `working_dir`, thrown before a single trial runs. One line turns this
replication off — harmless here, since every worker already shares the driver's environment
on a single machine, which is all `ray.tune` needs for local hypertuning.

In [ ]:
import os
os.environ.setdefault("RAY_ENABLE_UV_RUN_RUNTIME_ENV", "0")

import ray
from ray import tune
from ray.tune.schedulers import AsyncHyperBandScheduler
from ray.tune.search.hyperopt import HyperOptSearch
from sklearn.metrics import f1_score
from vectormesh.components import (
    AttentionAggregator, MaskedMeanAggregator, MeanAggregator, NeuralNet,
    Projection, RNNAggregator, Serial, TransformerBlock,
)

D_IN = meta["hidden_size"]

AGGREGATORS = {
    "mean": lambda d: MeanAggregator(),
    "masked_mean": lambda d: MaskedMeanAggregator(),
    "attention": lambda d: AttentionAggregator(hidden_size=d),
    "rnn": lambda d: RNNAggregator(hidden_size=d),
}


def build_pipeline(config):
    parts = [Projection(hidden_size=D_IN, out_size=config["bottleneck_dim"])]
    for _ in range(config["n_transformer_blocks"]):
        parts.append(TransformerBlock(hidden_size=config["bottleneck_dim"],
                                      num_heads=config["num_heads"]))
    parts.append(AGGREGATORS[config["aggregator"]](config["bottleneck_dim"]))
    parts.append(NeuralNet(hidden_size=config["bottleneck_dim"], out_size=K))
    return Serial(parts)

The trainable function trains that pipeline and reports **every epoch**, not just the
last one — the scheduler below needs a partial curve to decide which trials are worth
continuing. Note how the data reaches each trial: `tune.with_parameters` puts `tune_train_ds`
and `tune_valid_ds` into Ray's object store **once**, and every trial gets a reference to the
same copy instead of a re-serialised one. L4's `01_hypertuner` used a `FileLock` for the same
problem in a different shape — there the data lived on disk and had to be *downloaded* once
per worker; here it already lives in memory, so this is the cheaper tool for the job.

In [ ]:
def train_fn(config, train_ds=None, valid_ds=None, epochs=10):
    from pathlib import Path

    from torch.utils.data import DataLoader
    from vectormesh.components import FixedPadding
    from vectormesh.components.metrics import Accuracy
    from vectormesh.data import Collate
    from vectormesh.training import Trainer, TrainerSettings

    assert train_ds is not None and valid_ds is not None

    collate = Collate(embedding_col="embed", target_col="onehot",
                      padder=FixedPadding(max_chunks=config["max_chunks"]))
    tr = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate, num_workers=0)
    va = DataLoader(valid_ds, batch_size=32, shuffle=False, collate_fn=collate, num_workers=0)

    model = build_pipeline(config)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    def to_ray(epoch, train_loss, test_loss, metric_dict):
        # f1_macro needs raw predictions, which metric_dict does not carry -- one
        # extra pass over the (small, 100-document) validation set per epoch.
        model.eval()
        preds, gold = [], []
        with torch.no_grad():
            for X, y in va:
                preds += model(X).argmax(-1).tolist()
                gold += y.argmax(-1).tolist()
        tune.report({
            "accuracy": metric_dict["Accuracy"],
            "f1_macro": f1_score(gold, preds, average="macro"),
            "params": n_params,
        })

    # vectormesh.training.Trainer: log_dir is computed but never created on disk
    # unless something writes to it, and progress=False drops the tqdm bars --
    # what makes constructing one Trainer per ray trial cheap.
    settings = TrainerSettings(
        epochs=epochs,
        metrics=[Accuracy()],
        logdir=Path("ray_results/trials").resolve(),
        train_steps=len(tr),
        valid_steps=len(va),
        earlystop_kwargs=None,
    )
    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=torch.nn.CrossEntropyLoss(),
        optimizer=torch.optim.Adam,
        traindataloader=tr,
        validdataloader=va,
        reporters=[to_ray],
        progress=False,
    )
    trainer.loop()

## 5.9 Run it: TPE proposes, Hyperband prunes

`01_hypertuner` walked through random search, grid search, Bayesian optimisation and
Hyperband one at a time, ending on the combination it calls "Hyperbayes": `HyperOptSearch`
(the Tree-structured Parzen Estimator, TPE — a Bayesian method that handles a mixed
categorical/discrete space like this one natively, which is exactly this search space) picks
configurations, and `AsyncHyperBandScheduler` kills the unpromising ones early instead of
running every trial to the full epoch budget. That combination is what runs here — no new
concepts, only a new space to point it at.

In [ ]:
from notebooktester import param

SEARCH_SPACE = {
    "aggregator": tune.choice(["mean", "masked_mean", "attention", "rnn"]),
    "max_chunks": tune.choice([4, 8, 16, 20, 24, 32]),
    "bottleneck_dim": tune.choice([32, 64, 128, 192, 256]),
    "n_transformer_blocks": tune.choice([0, 1, 2, 3]),
    "num_heads": tune.choice([1, 2, 4, 8])
}
EPOCHS, N_TRIALS = param(15, test=1), param(100, test=1)

TUNE_DIR = ARTEFACTS / "scotus_tune"

search_alg = HyperOptSearch(metric="accuracy", mode="max")
# scheduler = AsyncHyperBandScheduler(
#     time_attr="training_iteration", grace_period=2, reduction_factor=3, max_t=EPOCHS)

t0 = time.time()
tuner = tune.Tuner(
    tune.with_parameters(train_fn, train_ds=tune_train_ds, valid_ds=tune_valid_ds,
                         epochs=EPOCHS),
    param_space=SEARCH_SPACE,
    tune_config=tune.TuneConfig(metric="accuracy", 
                                mode="max", 
                                search_alg=search_alg,
                                # scheduler=scheduler, 
                                num_samples=N_TRIALS),
    run_config=tune.RunConfig(storage_path=str(TUNE_DIR), verbose=1),
)
search_results = tuner.fit()
print(f"\n{N_TRIALS} trials in {time.time() - t0:.0f}s")

## 5.10 Reading the search — carefully

The obvious next cell is a scatter plot and a "winner". Both are worth having, but neither is
the point of this section: the point is that a search's own leftovers are **not** an ablation,
and mistaking one for the other is the same error 06_composition's C10 correction spent a
whole notebook undoing — just one level up.

In [ ]:
import plotly.express as px

df = search_results.get_dataframe()
best = search_results.get_best_result()
assert best.metrics is not None and best.config is not None
print("best config:", best.config)
print(f"  -> accuracy {best.metrics['accuracy']:.3f}, "
      f"f1_macro {best.metrics['f1_macro']:.3f}, "
      f"{best.metrics['params']:,d} params")


final = df.sort_values("accuracy", ascending=False)
cols = ["config/aggregator", "config/max_chunks", "config/n_transformer_blocks",
        "config/bottleneck_dim", "config/num_heads", "accuracy"]

fig = px.parallel_categories(final[cols],
                              color="accuracy",
                              color_continuous_scale=px.colors.sequential.Viridis)

numeric_cols = {"config/max_chunks", "config/bottleneck_dim", "config/num_heads", "config/n_transformer_blocks"}

for dim in fig.data[0].dimensions:
    if dim.label in numeric_cols:
        dim.categoryorder = "category ascending"

fig.show()

## 5.11 The number that counts: one run, on data the search never saw

Retrain the winning configuration on the **full** 600-document pool (tuning no longer needs a
held-out validation split — that job is done) for a longer budget, then evaluate **once** on
the 300-document test cache. This is the only number in the notebook that is allowed to go in
a report.

In [ ]:
from vectormesh.components.metrics import Accuracy
from vectormesh.training import Trainer, TrainerSettings

FINAL_EPOCHS = 20

final_loader_train = DataLoader(
    train_oh, batch_size=32, shuffle=True, num_workers=0,
    collate_fn=Collate(embedding_col="embed", target_col="onehot",
                       padder=FixedPadding(max_chunks=best.config["max_chunks"])))
final_loader_test = DataLoader(
    test_oh, batch_size=32, shuffle=False, num_workers=0,
    collate_fn=Collate(embedding_col="embed", target_col="onehot",
                       padder=FixedPadding(max_chunks=best.config["max_chunks"])))

torch.manual_seed(0)
final_model = build_pipeline(best.config)

settings = TrainerSettings(
    epochs=FINAL_EPOCHS,
    metrics=[Accuracy()],
    logdir=Path("ray_results/final").resolve(),
    train_steps=len(final_loader_train),
    valid_steps=len(final_loader_test),
    earlystop_kwargs=None,
)
final_trainer = Trainer(
    model=final_model,
    settings=settings,
    loss_fn=torch.nn.CrossEntropyLoss(),
    optimizer=torch.optim.Adam,
    traindataloader=final_loader_train,
    validdataloader=final_loader_test,
    device=str(device),
)
t0 = time.time()
final_trainer.loop()
seconds = time.time() - t0

final_model.eval()
preds, gold = [], []
with torch.no_grad():
    for X, y in final_loader_test:
        preds += final_model(X.to(device)).argmax(-1).cpu().tolist()
        gold += y.argmax(-1).tolist()

test_acc = sum(p == g for p, g in zip(preds, gold)) / len(gold)
test_f1 = f1_score(gold, preds, average="macro")
print(f"config: {best.config}")
print(f"validation, during the search: accuracy {best.metrics['accuracy']:.3f}")
print(f"held-out test, never seen by the search: accuracy {test_acc:.3f}, "
      f"f1_macro {test_f1:.3f}  ({seconds:.0f}s, {FINAL_EPOCHS} epochs, {len(train_oh)} docs)")

In [ ]:
gap = best.metrics["accuracy"] - test_acc
agg_means = final.groupby("config/aggregator")["accuracy"].mean().sort_values(ascending=False)
winner_rank = list(agg_means.index).index(best.config["aggregator"]) + 1

print(f"validation -> test gap: {gap:+.3f}")
print(f"\nmean validation accuracy by aggregator, best first:")
print(agg_means)
print(f"\nthe search's winning trial used '{best.config['aggregator']}', which ranks "
      f"#{winner_rank} of {len(agg_means)} aggregators **by average** validation accuracy.")

if gap > 0.03 and winner_rank > 1:
    print("\nThat is the story: a single trial beat the odds on 100 validation documents,")
    print("using an aggregator that is not even the best *on average*. The search picked the")
    print("lucky trial, not the good architecture, and the gap above is what that luck cost")
    print("once measured against data it never got to fit.")
elif gap > 0.03:
    print("\nThe winner used the best-average aggregator and still dropped on held-out data —")
    print("100 validation documents is a noisy estimate even for the best group.")
else:
    print("\nThe validation and test numbers roughly agree here — but that is a property of")
    print("this particular run, not a guarantee. Rerun the search (it is not seeded on purpose,")
    print("see the note below) and check whether the gap stays small.")

**Why the search above is not seeded.** It would be easy to fix `HyperOptSearch`'s
`random_state_seed` so this cell always finds the same "winner" and this paragraph could
quote one fixed pair of numbers. That would make the notebook reproducible and would also
quietly teach the wrong lesson — that a hyperparameter search produces *a* correct answer to
memorise, rather than an estimate with a validation set's worth of noise attached to it. Rerun
§5.9 a few times and watch the winning config move around; that instability is the actual
content of this section, not a defect in the demo. What must be reproducible is the **method**
— tune on validation, confirm on a test set the search never touched — not the specific
numbers any one run produces.

The general shape to expect: with 100 validation documents split across 13 classes, some of
them with only a handful of examples, 40 trials have real room to find a fluke. That is not a
reason to distrust `ray.tune` — it is a reason the held-out test evaluation in this section
exists at all, and it is the same leakage discipline lesson 3 teaches about time-series splits,
now showing up one level up the pipeline: at the level of *which hyperparameters you trust*,
not just which rows you trained on.

## Exercises

**E1 — turn the search's best guess into a controlled ablation.** Pick the axis §5.10's table
made you most confident about (e.g. "bottleneck 256 beats 32"). Fix every other hyperparameter
to the winning config and retrain at just that one axis's candidate values, three seeds each,
the way `06_composition` does. Does the ranking survive being isolated? This is the exercise
that answers whether the search's leftovers were telling the truth.

**E2 — race the search algorithms.** `01_hypertuner` compares random search, grid search,
Bayesian optimisation and Hyperband on one problem. Run at least two of them here, on *this*
search space, budget-matched (same `N_TRIALS` and `EPOCHS`). Plot best-accuracy-so-far against
wall-clock for each. Which one would you actually wait for?

**E3 — is 40 trials enough?** Rerun with `N_TRIALS=100`. Does the winning configuration
change? Does the parallel-categories plot look meaningfully different, or does it just get
more crowded in the same region?

**E4 — fixed or dynamic?** The search always used `FixedPadding`. Swap in `DynamicPadding()`
for the winning `max_chunks` value and compare. Which documents does each strategy waste, and
why does the choice matter more here than it did on IMDB in lesson 1?

**E5 — does the vectormesh warning hold?** vectormesh's docs (§5.5) warn that
`MaskedMeanAggregator` becomes a no-op after a `TransformerBlock`, because the block's
`LayerNorm` destroys the all-zero signature padding relies on. Filter `df` to
`use_transformer == True` and compare `aggregator == "masked_mean"` against
`aggregator == "mean"`. Are they indistinguishable, as the warning predicts?